In [ ]:
!pip install transformers torch scikit-learn matplotlib --quiet

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
from sklearn.decomposition import PCA
import itertools
import matplotlib.pyplot as plt



MODEL_NAME = "google/gemma-2-2b-it"

# 1. Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()
emb_matrix = model.get_input_embeddings().weight.detach().cpu().numpy()

# 2. Helper: get embedding for a word
def best_tokenization(word):
    candidates = [word, " " + word] if not word.startswith(" ") else [word]
    best = None
    for cand in candidates:
        ids = tokenizer.encode(cand, add_special_tokens=False)
        if best is None or len(ids) < len(best[1]):
            best = (cand, ids)
    return best

def embedding_for_word(word, aggregation="mean"):
    _, ids = best_tokenization(word)
    vecs = emb_matrix[np.array(ids)]
    if len(ids) == 1:
        return vecs[0], ids
    else:
        return vecs.mean(axis=0), ids

# 3. Collect embeddings
targets = ["privacy", "important", "overrated"]
records = []
for w in targets:
    vec, ids = embedding_for_word(w)
    tokens = [tokenizer.decode([i]) for i in ids]
    records.append({"word": w, "vec": vec, "tokens": tokens, "ids": ids})

# 4. PCA to 2D
X = np.vstack([r["vec"] for r in records])
pca = PCA(n_components=2, random_state=0)
X2 = pca.fit_transform(X)
for r, x2 in zip(records, X2):
    r["pca2"] = x2

# 5. Compute all pairwise dot products (3×3 matrix)
def dot_matrix(vectors):
    n = len(vectors)
    mat = np.zeros((n, n))
    for i, j in itertools.product(range(n), range(n)):
        mat[i, j] = np.dot(vectors[i], vectors[j])
    return mat

dot_high = dot_matrix([r["vec"] for r in records])
dot_low  = dot_matrix([r["pca2"] for r in records])

# 6. Print summary tables
print(f"Model: {MODEL_NAME}")
print(f"Embedding dimension: {emb_matrix.shape[1]}\n")

print("=== High-dimensional GPT-2 embeddings (first 8 dims) ===")
for r in records:
    print(f"{r['word']:>6s}: {np.round(r['vec'][:8],6)} ...")

print("\n=== Dot products between GPT-2 embeddings ===")
print(f"{'':8s}" + "".join(f"{w:>12s}" for w in targets))
for i, wi in enumerate(targets):
    print(f"{wi:8s}" + "".join(f"{dot_high[i,j]:12.4f}" for j in range(len(targets))))

print("\n=== 2D PCA coordinates ===")
for r in records:
    print(f"{r['word']:>6s}: ({r['pca2'][0]:.6f}, {r['pca2'][1]:.6f})")

print("\n=== Dot products between 2D PCA coordinates ===")
print(f"{'':8s}" + "".join(f"{w:>12s}" for w in targets))
for i, wi in enumerate(targets):
    print(f"{wi:8s}" + "".join(f"{dot_low[i,j]:12.4f}" for j in range(len(targets))))

# 7. Sort the 9 dot products for each space
def sorted_dot_products(matrix, labels):
    """Return sorted list of (pair_label, value) for all i,j"""
    n = len(labels)
    pairs = []
    for i, j in itertools.product(range(n), range(n)):
        pairs.append((f"{labels[i]}·{labels[j]}", matrix[i,j]))
    # Sort descending by dot-product value
    pairs.sort(key=lambda x: x[1], reverse=True)
    return pairs

sorted_high = sorted_dot_products(dot_high, targets)
sorted_low  = sorted_dot_products(dot_low, targets)

print("\n=== Ordered dot products (high-dimensional GPT-2 embeddings) ===")
for name, val in sorted_high:
    print(f"{name:>12s}: {val: .6f}")

print("\n=== Ordered dot products (2D PCA projections) ===")
for name, val in sorted_low:
    print(f"{name:>12s}: {val: .6f}")

# 8. 2D scatter plot
plt.figure(figsize=(5,5))
for r in records:
    plt.scatter(r["pca2"][0], r["pca2"][1], s=100)
    plt.text(r["pca2"][0]+0.02, r["pca2"][1]+0.02, r["word"], fontsize=12)
plt.title(f"GPT-2 ({MODEL_NAME}) Embeddings → PCA(2D)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(True)
plt.show()


The next part does this: Drop-in code patch (adds norms + cosine matrices + sorted cosine lists)

In [ ]:
import numpy as np
import itertools

# --- Norms from the diagonal ---
norms = {w: np.sqrt(dot_high[i, i]) for i, w in enumerate(["earth","flat","round"])}
print("\n=== Vector norms (from self-dots) ===")
for w, n in norms.items():
    print(f"{w:>6s}: {n:.6f}")

# --- Cosine similarity matrix in 768-D ---
def cosine_matrix(vectors):
    V = np.vstack(vectors)                        # shape (n, d)
    n = V.shape[0]
    norms = np.linalg.norm(V, axis=1, keepdims=True) + 1e-12
    U = V / norms
    return U @ U.T

cos_high = cosine_matrix([r["vec"] for r in records])

print("\n=== Cosine similarities (768-D embeddings) ===")
labels = ["earth","flat","round"]
print(f"{'':8s}" + "".join(f"{w:>12s}" for w in labels))
for i, wi in enumerate(labels):
    print(f"{wi:8s}" + "".join(f"{cos_high[i,j]:12.4f}" for j in range(len(labels))))

# --- Sorted cosines (largest -> smallest), include diagonal for reference ---
pairs = []
for i, j in itertools.product(range(len(labels)), range(len(labels))):
    pairs.append((f"{labels[i]}·{labels[j]}", float(cos_high[i, j])))

pairs.sort(key=lambda x: x[1], reverse=True)
print("\n=== Ordered cosine similarities (768-D) ===")
for name, val in pairs:
    print(f"{name:>12s}: {val: .6f}")

# --- Optional: cosines for the 2D PCA points, too ---
cos_low = cosine_matrix([r["pca2"] for r in records])
pairs2 = []
for i, j in itertools.product(range(len(labels)), range(len(labels))):
    pairs2.append((f"{labels[i]}·{labels[j]}", float(cos_low[i, j])))
pairs2.sort(key=lambda x: x[1], reverse=True)
print("\n=== Ordered cosine similarities (2D PCA) ===")
for name, val in pairs2:
    print(f"{name:>12s}: {val: .6f}")